# Inspect ANN Index: Recall Comparison vs kNN

After running `manage_ann_index.py` (step 10), this notebook inspects the ANN index on the combined collection and compares ANN search results against kNN ground truth to measure recall. It also demonstrates `SearchHint` usage:

1. **Index status** — List indexes, inspect configuration (field, distance metric, filter_fields, store_fields)
2. **Collection size** — Count DataObjects vs ANN threshold
3. **SearchHint** — How to control ANN vs kNN at query time
4. **kNN baseline** — Brute-force search with `SearchHint(use_knn=True)`
5. **ANN search** — Default search (auto-selects ANN if available)
6. **Recall comparison** — Measure `recall@k = |ANN ∩ kNN| / k` across test queries

---
## Setup

In [1]:
import time

import pandas as pd
from google.cloud import vectorsearch_v1beta
from google import genai
from config import (
    PROJECT_ID, REGION, EMBEDDING_MODEL, EMBEDDING_DIMENSIONS,
    VS_COLLECTION_COMBINED,
    ANN_INDEX_THRESHOLD, ANN_INDEX_ID,
)

vs_client = vectorsearch_v1beta.VectorSearchServiceClient()
search_client = vectorsearch_v1beta.DataObjectSearchServiceClient()
genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)

COMBINED = f'projects/{PROJECT_ID}/locations/{REGION}/collections/{VS_COLLECTION_COMBINED}'

print(f'Collection: {VS_COLLECTION_COMBINED}')
print(f'ANN index ID: {ANN_INDEX_ID}')
print(f'ANN threshold: {ANN_INDEX_THRESHOLD}')

Collection: applied-genai-solutions-multi-type-parsing-combined
ANN index ID: dense-embedding
ANN threshold: 1000


---
## Index Status

List all indexes on the combined collection and inspect their configuration.

In [2]:
indexes = list(vs_client.list_indexes(parent=COMBINED))

if not indexes:
    print('No ANN indexes found on the combined collection.')
    print(f'Run manage_ann_index.py to create one (requires >= {ANN_INDEX_THRESHOLD} DataObjects).')
    ann_available = False
else:
    ann_available = True
    for idx in indexes:
        idx_id = idx.name.split('/indexes/')[-1] if '/indexes/' in idx.name else idx.name
        print(f'Index: {idx_id}')
        print(f'  Field: {idx.index_field}')
        print(f'  Distance metric: {idx.distance_metric}')
        print(f'  Filter fields: {list(idx.filter_fields)}')
        print(f'  Store fields: {list(idx.store_fields)}')
        print(f'  Display name: {idx.display_name}')
        print(f'  Name: {idx.name}')

No ANN indexes found on the combined collection.
Run manage_ann_index.py to create one (requires >= 1000 DataObjects).


---
## Collection Size

In [3]:
count_response = search_client.aggregate_data_objects(
    vectorsearch_v1beta.AggregateDataObjectsRequest(
        parent=COMBINED, aggregate='COUNT',
    )
)
data_object_count = int(count_response.aggregate_results[0]['count'])

print(f'DataObjects: {data_object_count}')
print(f'ANN threshold: {ANN_INDEX_THRESHOLD}')
if data_object_count >= ANN_INDEX_THRESHOLD:
    print(f'Collection meets threshold for ANN index creation.')
else:
    print(f'Below threshold ({data_object_count} < {ANN_INDEX_THRESHOLD}). Brute-force kNN is used for all queries.')

DataObjects: 117
ANN threshold: 1000
Below threshold (117 < 1000). Brute-force kNN is used for all queries.


---
## SearchHint: Controlling ANN vs kNN

`SearchHint` is available on `VectorSearch` and `SemanticSearch` (not `TextSearch`). It controls which search engine is used:

| Behavior | How to set | When to use |
|----------|-----------|-------------|
| **Auto (default)** | Omit `search_hint` | Production queries — uses ANN index if available, falls back to kNN |
| **Force kNN** | `search_hint=SearchHint(use_knn=True)` | Recall evaluation — brute-force exact results as ground truth |
| **Force specific index** | `search_hint=SearchHint(use_index=IndexHint(name=...))` | Multi-index collections — target a specific ANN index |

`SearchResponseMetadata` in the response reports which engine was used (`used_index` or `used_knn`).

**At small scale** (117 DataObjects), VS2 may use kNN even with an ANN index present, since brute-force is efficient at this size. ANN provides latency benefits at thousands+ of DataObjects.

---
## kNN Baseline

Run SemanticSearch with `SearchHint(use_knn=True)` to force brute-force kNN. This produces exact nearest neighbor results — the ground truth for recall comparison.

In [4]:
query = 'What machine learning methods work best for demand forecasting?'
top_k = 10

knn_response = search_client.search_data_objects(
    vectorsearch_v1beta.SearchDataObjectsRequest(
        parent=COMBINED,
        semantic_search=vectorsearch_v1beta.SemanticSearch(
            search_text=query,
            search_field='text_content_embedding',
            task_type='QUESTION_ANSWERING',
            top_k=top_k,
            output_fields=vectorsearch_v1beta.OutputFields(
                data_fields=['chunk_id', 'source_type', 'text_content']
            ),
            search_hint=vectorsearch_v1beta.SearchHint(use_knn=True),
        ),
    )
)

knn_results = list(knn_response.results)
knn_ids = [r.data_object.data['chunk_id'] for r in knn_results]

print(f'kNN search: "{query}"')
print(f'Results: {len(knn_results)}')
print(f'Metadata: {knn_response.search_response_metadata}')
print()
for rank, r in enumerate(knn_results, 1):
    data = r.data_object.data
    print(f'  {rank}. [{data["source_type"]}] {data["chunk_id"]}: {data["text_content"][:100]}...')

kNN search: "What machine learning methods work best for demand forecasting?"
Results: 10
Metadata: 

  1. [reddit] red_thread_deep_015: THREAD: ARIMA vs. Prophet vs. LightGBM for Demand Forecasting - What's your go-to and why? | PARENT:...
  2. [reddit] red_thread_shallow_003: THREAD: ARIMA vs. Prophet for Retail Demand Forecasting - What's your go-to in 2024? | COMMENT: Skip...
  3. [reddit] red_thread_medium_000: THREAD: Beyond ARIMA and Prophet: What are your go-to advanced time series forecasting methods for c...
  4. [reddit] red_thread_deep_016: THREAD: ARIMA vs. Prophet vs. LightGBM for Demand Forecasting - What's your go-to and why? | PARENT:...
  5. [reddit] red_thread_medium_001: THREAD: Beyond ARIMA and Prophet: What are your go-to advanced time series forecasting methods for c...
  6. [zoom] zoom_meeting_model_planning_000: Sarah (Lead): Alright everyone, thanks for joining. Let's kick off. The main agenda item for today i...
  7. [reddit] red_thread_shallow_004: THREAD: A

---
## ANN Search (Default)

Run the same query without `search_hint` — VS2 auto-selects the ANN index if one exists, otherwise falls back to kNN. Check `search_response_metadata` to confirm which engine was used.

In [5]:
default_response = search_client.search_data_objects(
    vectorsearch_v1beta.SearchDataObjectsRequest(
        parent=COMBINED,
        semantic_search=vectorsearch_v1beta.SemanticSearch(
            search_text=query,
            search_field='text_content_embedding',
            task_type='QUESTION_ANSWERING',
            top_k=top_k,
            output_fields=vectorsearch_v1beta.OutputFields(
                data_fields=['chunk_id', 'source_type', 'text_content']
            ),
        ),
    )
)

default_results = list(default_response.results)
default_ids = [r.data_object.data['chunk_id'] for r in default_results]

print(f'Default search: "{query}"')
print(f'Results: {len(default_results)}')
print(f'Metadata: {default_response.search_response_metadata}')
print()
for rank, r in enumerate(default_results, 1):
    data = r.data_object.data
    print(f'  {rank}. [{data["source_type"]}] {data["chunk_id"]}: {data["text_content"][:100]}...')

# Quick recall check for this single query
overlap = set(default_ids[:top_k]) & set(knn_ids[:top_k])
recall = len(overlap) / top_k
print(f'\nrecall@{top_k}: {recall:.2f} ({len(overlap)}/{top_k} results match kNN)')

Default search: "What machine learning methods work best for demand forecasting?"
Results: 10
Metadata: 

  1. [reddit] red_thread_deep_015: THREAD: ARIMA vs. Prophet vs. LightGBM for Demand Forecasting - What's your go-to and why? | PARENT:...
  2. [reddit] red_thread_shallow_003: THREAD: ARIMA vs. Prophet for Retail Demand Forecasting - What's your go-to in 2024? | COMMENT: Skip...
  3. [reddit] red_thread_medium_000: THREAD: Beyond ARIMA and Prophet: What are your go-to advanced time series forecasting methods for c...
  4. [reddit] red_thread_deep_016: THREAD: ARIMA vs. Prophet vs. LightGBM for Demand Forecasting - What's your go-to and why? | PARENT:...
  5. [reddit] red_thread_medium_001: THREAD: Beyond ARIMA and Prophet: What are your go-to advanced time series forecasting methods for c...
  6. [zoom] zoom_meeting_model_planning_000: Sarah (Lead): Alright everyone, thanks for joining. Let's kick off. The main agenda item for today i...
  7. [reddit] red_thread_shallow_004: THREA

---
## Recall Comparison

Run a set of test queries through both kNN (ground truth) and default (ANN if available) search, then compute `recall@k`:

```
recall@k = |ANN_results ∩ kNN_results| / k
```

At small scale (117 DataObjects), ANN and kNN should produce identical results (recall = 1.0). As the collection grows to thousands+, ANN may trade a small amount of recall for significant latency improvement.

In [6]:
test_queries = [
    'What machine learning methods work best for demand forecasting?',
    'How does ARIMA handle seasonal patterns?',
    'Prophet vs LightGBM for time series',
    'How to create a forecasting model in BigQuery ML?',
    'What are the limitations of ARIMA PLUS models?',
]

k_values = [5, 10]
rows = []

for q in test_queries:
    # kNN baseline
    t0 = time.time()
    knn_resp = search_client.search_data_objects(
        vectorsearch_v1beta.SearchDataObjectsRequest(
            parent=COMBINED,
            semantic_search=vectorsearch_v1beta.SemanticSearch(
                search_text=q,
                search_field='text_content_embedding',
                task_type='QUESTION_ANSWERING',
                top_k=max(k_values),
                output_fields=vectorsearch_v1beta.OutputFields(
                    data_fields=['chunk_id']
                ),
                search_hint=vectorsearch_v1beta.SearchHint(use_knn=True),
            ),
        )
    )
    knn_latency = time.time() - t0
    knn_ids = [r.data_object.data['chunk_id'] for r in knn_resp.results]

    # Default (ANN if available)
    t0 = time.time()
    default_resp = search_client.search_data_objects(
        vectorsearch_v1beta.SearchDataObjectsRequest(
            parent=COMBINED,
            semantic_search=vectorsearch_v1beta.SemanticSearch(
                search_text=q,
                search_field='text_content_embedding',
                task_type='QUESTION_ANSWERING',
                top_k=max(k_values),
                output_fields=vectorsearch_v1beta.OutputFields(
                    data_fields=['chunk_id']
                ),
            ),
        )
    )
    default_latency = time.time() - t0
    default_ids = [r.data_object.data['chunk_id'] for r in default_resp.results]

    row = {'query': q[:60] + '...' if len(q) > 60 else q}
    for k in k_values:
        overlap = set(default_ids[:k]) & set(knn_ids[:k])
        row[f'recall@{k}'] = len(overlap) / k
    row['knn_ms'] = f'{knn_latency * 1000:.0f}'
    row['default_ms'] = f'{default_latency * 1000:.0f}'
    row['engine'] = str(default_resp.search_response_metadata)
    rows.append(row)

df = pd.DataFrame(rows)
print('Recall comparison: default (ANN if available) vs kNN ground truth')
print(f'Collection: {data_object_count} DataObjects\n')
df

Recall comparison: default (ANN if available) vs kNN ground truth
Collection: 117 DataObjects



,query,recall@5,recall@10,knn_ms,default_ms,engine
0,What machine learning methods work best for de...,1.0,1.0,251,205,
1,How does ARIMA handle seasonal patterns?,1.0,1.0,123,106,
2,Prophet vs LightGBM for time series,1.0,1.0,100,137,
3,How to create a forecasting model in BigQuery ML?,1.0,1.0,215,197,
4,What are the limitations of ARIMA PLUS models?,1.0,1.0,139,154,


---
## Observations

- **At small scale** (117 DataObjects), ANN and kNN produce identical results (recall = 1.0). VS2 may even use kNN internally when the collection is small enough that brute-force is faster.
- **As the collection grows** to thousands+ of DataObjects, ANN trades a small amount of recall for significantly lower query latency. The latency gap becomes more pronounced at larger scales.
- **Use kNN for evaluation/ground truth** — `SearchHint(use_knn=True)` gives exact nearest neighbors, useful for measuring retrieval quality.
- **Use ANN (default) for production queries** — the latency improvement at scale is worth the negligible recall trade-off.
- **The ANN index auto-updates** as DataObjects are added or removed — no manual rebuild needed.